# Chapter 6 — Runtime State, Progress, and Termination

**Book alignment:** current Chapter 6 · internal demo `Stage 05`

The shared demo package calls this **Stage 05** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** Can the runtime distinguish progress from activity, recover from stalls, and stop for named reasons under explicit budgets?


## Hypothesis

A loop should continue, recover, or stop from **state transitions and progress**, not mere activity. Repeating an action is acceptable when it produces new evidence and suspicious when the same relevant state repeats without useful change.


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'demo' / 'agents-from-first-principles').exists():
            return candidate
    raise RuntimeError('Run this notebook from a checkout containing demo/agents-from-first-principles')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / 'demo' / 'agents-from-first-principles'
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent.runtime import (
    Budgets,
    ContinuationDisposition,
    ContinuationPolicy,
    ExecutionTrace,
    LongTermMemory,
    ModelVisibleContext,
    ProgressMeasure,
    RuntimeState,
    StepRecord,
    TerminationReason,
)


## Controlled experiment: activity versus progress

The same action can be productive or nonproductive depending on what it changes or reveals.


In [ ]:
policy = ContinuationPolicy()
budgets = Budgets(max_steps=10, max_model_calls=10, no_progress_window=2)

stagnant = ExecutionTrace([
    StepRecord('read_file parser.py', 'parser unchanged', new_evidence=True, progress_delta=1),
    StepRecord('read_file parser.py', 'parser unchanged', new_evidence=False, progress_delta=0),
])

productive_repeat = ExecutionTrace([
    StepRecord('run_tests', 'one failure', new_evidence=True, progress_delta=1),
    StepRecord('run_tests', 'different failure detail', new_evidence=True, progress_delta=1),
])

plateau = ExecutionTrace([
    StepRecord('inspect', 'state A', new_evidence=True, progress_delta=1),
    StepRecord('inspect', 'state B'),
    StepRecord('inspect', 'state C'),
])

stagnant_decision = policy.decide(RuntimeState(step_count=2, model_calls=2), stagnant, budgets)
productive_decision = policy.decide(RuntimeState(step_count=2, model_calls=2), productive_repeat, budgets)
plateau_decision = policy.decide(RuntimeState(step_count=3, model_calls=3), plateau, budgets)

{
    'stagnant': (stagnant_decision.disposition.value, stagnant_decision.reason.value),
    'productive_repeat': (productive_decision.disposition.value, productive_decision.reason),
    'plateau': (plateau_decision.disposition.value, plateau_decision.reason.value),
    'steps_after_last_progress': ProgressMeasure.steps_after_last_progress(plateau),
}


## Assertions


In [ ]:
assert stagnant_decision.disposition is ContinuationDisposition.RECOVER
assert stagnant_decision.reason is TerminationReason.NONPRODUCTIVE_CYCLE

assert productive_decision.disposition is ContinuationDisposition.CONTINUE
assert productive_decision.reason is None

assert plateau_decision.disposition is ContinuationDisposition.RECOVER
assert plateau_decision.reason is TerminationReason.NO_PROGRESS
assert plateau_decision.steps_after_last_progress == 2

print({
    'stagnant': stagnant_decision.reason.value,
    'productive_repeat': productive_decision.disposition.value,
    'steps_after_last_progress': plateau_decision.steps_after_last_progress,
})


## Experiment: progress is a claim about state, not a reward token

The current chapter makes a sharper point than the original notebook: once continuation depends on a progress signal, that signal becomes a target. If a caller can label repeated activity as `new_evidence=True`, the Stage-05 controller will trust the label and continue.

This is an intentional ablation. It shows why progress accounting needs **integrity constraints tied to observations/state transitions**, rather than self-reported activity.


In [ ]:
gamed_progress = ExecutionTrace([
    StepRecord('read_file parser.py', 'parser unchanged', new_evidence=True, progress_delta=1),
    StepRecord('read_file parser.py', 'parser unchanged', new_evidence=True, progress_delta=1),
])

gamed_decision = policy.decide(
    RuntimeState(step_count=2, model_calls=2),
    gamed_progress,
    budgets,
)

assert gamed_decision.disposition is ContinuationDisposition.CONTINUE
assert gamed_decision.reason is None

{
    'same_activity_repeated': True,
    'self_reported_new_evidence': True,
    'controller_decision': gamed_decision.disposition.value,
}


## Independent budgets

Steps and model calls are different resources. Exhausting either budget can stop the runtime independently.


In [ ]:
by_steps = policy.decide(
    RuntimeState(step_count=5, model_calls=1),
    ExecutionTrace(),
    Budgets(max_steps=5, max_model_calls=10),
)
by_model_calls = policy.decide(
    RuntimeState(step_count=1, model_calls=4),
    ExecutionTrace(),
    Budgets(max_steps=10, max_model_calls=4),
)

assert by_steps.reason is TerminationReason.MAX_STEPS
assert by_model_calls.reason is TerminationReason.MAX_MODEL_CALLS
(by_steps.reason.value, by_model_calls.reason.value)


## State is not context, trace, or memory

These objects can contain related information without becoming interchangeable.


In [ ]:
runtime_state = RuntimeState(facts=frozenset({'parser unchanged'}))
model_visible_context = ModelVisibleContext(
    summary='parser still appears to use a fixed comma delimiter',
    facts=('parser unchanged',),
)
execution_trace = stagnant
long_term_memory = LongTermMemory()

assert runtime_state is not model_visible_context
assert execution_trace is not long_term_memory
assert runtime_state.facts == frozenset({'parser unchanged'})
assert long_term_memory.entries == ()

type(runtime_state).__name__, type(model_visible_context).__name__, type(execution_trace).__name__, type(long_term_memory).__name__


## Success is only a runtime signal here

Stage 05 can stop when it receives a success signal, but it does **not** define how success becomes true. Notebook 10 / Chapter 10 earns evidence-backed verification.


In [ ]:
success_decision = policy.decide(
    RuntimeState(success_signal=True),
    ExecutionTrace(),
    Budgets(),
)

assert success_decision.reason is TerminationReason.SUCCESS
assert 'Stage 09' in (success_decision.detail or '')
success_decision


## What was earned

The runtime can now distinguish productive work from activity, measure distance from the last progress event, enforce independent budgets, and return named `CONTINUE / RECOVER / STOP` decisions.

The final success oracle has **not** been introduced.


## Demo API now implemented

```python
from first_principles_agent.runtime import (
    RuntimeState,
    ExecutionTrace,
    ProgressMeasure,
    Budgets,
    ContinuationPolicy,
    ContinuationDecision,
    TerminationReason,
)

continuation.decide(state, trace, budgets) -> ContinuationDecision
```

Notebook 07 / Chapter 7 adds **capability contracts and action-space design** around the existing Stage-01 execution boundary.
